In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models

In [9]:
!pip install ncps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 1.9 MB/s eta 0:00:00


In [10]:
import pytorch_lightning as pl
import ncps

In [2]:
backbone = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
feature_extractor = backbone.features

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 89.7MB/s]


In [14]:
print(feature_extractor)

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
    (2): Hardswish()
  )
  (1): InvertedResidual(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (1): Conv2dNormActivation(
        (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      )
    )
  )
  (2): InvertedResidual(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=0.001, momentum=0.01, affine

In [13]:
batch_size = 256 #placeholder well decide the actual value later
image_h=256
image_w=384
S=8
image_feature_size=960
hidden_size=512
backbone_units=1024
ts = torch.ones(batch_size)

In [23]:
from ncps.torch import CfCCell
from ncps.torch.cfc import CfC
class GazeLLNArch(pl.LightningModule):
  def __init__(self,image_h,image_w,S,hidden_size,backbone_units):
    super().__init__()
    self.save_hyperparameters()

    self.image_h = image_h
    self.image_w = image_w
    self.S = S
    self.hidden_size = hidden_size
    self.backbone_units = backbone_units

    self.hmap_h = self.image_h // self.S
    self.hmap_w = self.image_w // self.S

    MobileNet = models.mobilenet_v3_large(weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    self.feature_extractor = MobileNet.features
    self.pool = nn.AdaptiveAvgPool2d(1)
    self.coordconv = CoordConv(in_channels = 1,out_channels = 1)
    self.image_feature_size = 960
    self.hmap_feature_size = 1*(self.hmap_h)*(self.hmap_w)

    cfc_input_size = self.image_feature_size + self.hmap_feature_size

    self.CfCCell = CfCCell(input_size = cfc_input_size,hidden_size = self.hidden_size,backbone_activation = "lecun_tanh",backbone_units=self.backbone_units,backbone_layers=1)
    self.project = nn.Linear(self.hidden_size,self.hmap_feature_size)

  def forward(self,image,prev_hmap,hx,ts):
    features = self.feature_extractor(image)
    features = self.pool(features).flatten(1)

    hmap_coords = self.coordconv(prev_hmap).flatten(1)

    x = torch.cat([features,hmap_coords],dim=1)
    x,hx = self.CfCCell(x,hx,ts)
    x = self.project(x)
    return x,hx

In [19]:

import torchvision.models as models

mobilenet = models.mobilenet_v3_large(weights=None)
features = mobilenet.features

dummy_image = torch.randn(2, 3, 256, 384)
out = features(dummy_image)
print("MobileNetV3 output:", out.shape)

MobileNetV3 output: torch.Size([2, 960, 8, 12])
